# 13 · Capstone: deliver a reproducible detection project

**Level:** advanced · **Class time:** 60–90 minutes · **Prerequisites:** 06–09; 12 for export.

**Brief:** detect and count three shape types for a small visual inspection demo.
Deliver annotated examples, per-image results, aggregate precision/recall, a saved
configuration and a short model report. The fully runnable baseline is a classical
shape detector; if notebook 07 produced a checkpoint, you can use the trained model.

**Presenter opening:** show an image, ask for a prediction, then expose the label and
the failure analysis. Do not describe synthetic validation as evidence of real-world readiness.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


## Step 1 — Freeze decisions before opening the test set

Use validation to choose confidence and IoU settings. The defaults below are fixed
teaching settings, not optimized thresholds. Set the engine now. Run the test once
for a final report. If you change settings after seeing test failures, those images
have become development data; collect a fresh test set for an unbiased final estimate.


In [ ]:
import os, json, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from course_utils import ensure_synthetic_dataset, audit_dataset, read_yolo_labels, CLASS_NAMES
DATA_YAML = ensure_synthetic_dataset()
DATA_ROOT = DATA_YAML.parent
ENGINE = "classical"  # Change to "yolo" after notebook 07 has saved a checkpoint.
CONFIDENCE = 0.25     # If tuned, use validation only and record the selection.
MATCH_IOU = 0.50
OUTPUT = ROOT / "artifacts" / "capstone"
OUTPUT.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame(audit_dataset(DATA_YAML)))


## Step 2 — Implement a small inference interface

Each engine returns pixel boxes, zero-based class IDs and scores. The classical
detector thresholds dark foreground, finds external contours and uses polygon
vertices to classify shapes. Its score is a fixed **placeholder ranking value** of 1,
not a learned confidence. This hand-built rule can fail when lighting or outlines change.


In [ ]:
def classical_predict(image):
    rgb = np.asarray(image.convert("RGB"))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    mask = np.uint8(gray < 205) * 255
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    found_boxes, found_classes = [], []
    for contour in contours:
        if cv2.contourArea(contour) < 30:
            continue
        perimeter = cv2.arcLength(contour, True)
        polygon = cv2.approxPolyDP(contour, 0.035 * perimeter, True)
        class_id = 2 if len(polygon) == 3 else 0 if len(polygon) == 4 else 1
        x, y, w, h = cv2.boundingRect(contour)
        found_boxes.append([x, y, x+w, y+h])
        found_classes.append(class_id)
    return (np.asarray(found_boxes, dtype=np.float32).reshape(-1, 4),
            np.asarray(found_classes, dtype=np.int64), np.ones(len(found_boxes)))

model = None
checkpoint = None
if ENGINE == "yolo":
    from ultralytics import YOLO
    manifest_path = ROOT / "artifacts" / "custom_model.json"
    if not manifest_path.exists():
        raise FileNotFoundError("Run notebook 07 first or select ENGINE='classical'.")
    checkpoint = json.loads(manifest_path.read_text())["best_weights"]
    if not Path(checkpoint).is_file():
        raise FileNotFoundError(f"Checkpoint missing: {checkpoint}")
    model = YOLO(checkpoint)
elif ENGINE != "classical":
    raise ValueError("Choose classical or yolo")

def predict(image):
    if model is None:
        return classical_predict(image)
    r = model.predict(image, conf=CONFIDENCE, imgsz=256, device="cpu", verbose=False)[0]
    return (r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int),
            r.boxes.conf.cpu().numpy())


## Step 3 — Match each prediction to at most one ground-truth box

Match in descending score order, on the same class, at IoU ≥ 0.5. A duplicate prediction
is a false positive; an unmatched ground truth is a false negative. A negative image
can have zero true positives and still be handled correctly. These aggregate counts
are **not mAP**; use notebook 08 / the framework evaluator for that metric.


In [ ]:
def iou_one_to_many(box, candidates):
    if not len(candidates):
        return np.zeros(0)
    lo = np.maximum(box[:2], candidates[:, :2])
    hi = np.minimum(box[2:], candidates[:, 2:])
    intersection = np.clip(hi-lo, 0, None).prod(axis=1)
    area = np.clip(box[2:]-box[:2], 0, None).prod()
    areas = np.clip(candidates[:, 2:]-candidates[:, :2], 0, None).prod(axis=1)
    return intersection / np.maximum(area+areas-intersection, 1e-9)

def match_image(pred_boxes, pred_classes, scores, gt_boxes, gt_classes, threshold):
    used, tp = set(), 0
    for index in np.argsort(-scores, kind="stable"):
        candidates = np.array([j for j, c in enumerate(gt_classes)
                               if c == pred_classes[index] and j not in used], dtype=int)
        overlap = iou_one_to_many(pred_boxes[index], gt_boxes[candidates])
        if overlap.size and overlap.max() >= threshold:
            used.add(int(candidates[int(overlap.argmax())]))
            tp += 1
    return tp, len(pred_boxes)-tp, len(gt_boxes)-tp


## Step 4 — Warm up, evaluate and save traceable results

The measured latency includes the inference function and its preprocessing but excludes
file loading, plotting and matching. This is not camera-to-screen latency. The sample
is small, so do not generalize timing or accuracy to another device or domain.


In [ ]:
# Warm up using a training image, before recording timings.
with Image.open(DATA_ROOT / "images/train/train_0001.png") as warmup_image:
    predict(warmup_image.convert("RGB"))
rows, examples = [], []
for path in sorted((DATA_ROOT / "images/test").glob("*.png")):
    image = Image.open(path).convert("RGB")
    gt_boxes, gt_classes = read_yolo_labels(DATA_ROOT / "labels/test" / (path.stem + ".txt"), *image.size)
    started = time.perf_counter()
    pb, pc, scores = predict(image)
    milliseconds = (time.perf_counter()-started)*1000
    tp, fp, fn = match_image(pb, pc, scores, gt_boxes, gt_classes, MATCH_IOU)
    rows.append({"image": path.name, "ground_truth_objects": len(gt_boxes), "predictions": len(pb),
                 "tp": tp, "fp": fp, "fn": fn, "latency_ms": milliseconds})
    if len(examples) < 6:
        examples.append((path.name, image.copy(), pb, pc, scores))
results = pd.DataFrame(rows)
results.to_csv(OUTPUT / "per_image_results.csv", index=False)
tp, fp, fn = results[["tp", "fp", "fn"]].sum()
summary = {"engine": ENGINE, "checkpoint": checkpoint, "split": "test", "images": len(results),
           "confidence": CONFIDENCE if ENGINE == "yolo" else None, "matching_iou": MATCH_IOU,
           "tp": int(tp), "fp": int(fp), "fn": int(fn),
           "precision": float(tp/(tp+fp)) if tp+fp else 0.0,
           "recall": float(tp/(tp+fn)) if tp+fn else 0.0,
           "median_latency_ms": float(results.latency_ms.median()),
           "p95_latency_ms": float(results.latency_ms.quantile(0.95)),
           "dataset": str(DATA_YAML), "scope": "Generated shapes only"}
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2))
display(pd.DataFrame([summary]))
display(results.sort_values(["fn", "fp"], ascending=False).head())


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, image, boxes, classes, scores) in zip(axes.flat, examples):
    show_boxes(image, boxes, [CLASS_NAMES[int(c)] for c in classes], scores=scores, ax=ax, title=name)
plt.tight_layout()
fig.savefig(OUTPUT / "predictions.png", dpi=140, bbox_inches="tight")
plt.show()


## Step 5 — Package the result with its limitations

A useful handoff contains the class mapping, dataset version, checkpoint, preprocessing,
threshold selection, metric definitions, environment and known failure cases. A model
file alone does not communicate those decisions. The template below records observed
metrics, not invented claims of production accuracy.


In [ ]:
from importlib.metadata import version
environment = {name: version(name) for name in ["numpy", "torch", "torchvision", "ultralytics"]}
(OUTPUT / "environment.json").write_text(json.dumps(environment, indent=2))
report = f"""# Shape detection demonstration

Engine: {ENGINE}
Classes: {', '.join(CLASS_NAMES)}
Dataset: procedural shapes, generator version 1; 96 train / 24 validation / 24 test.
Evaluation: class-aware, one-to-one matches at IoU {MATCH_IOU}; not COCO mAP.
Observed precision: {summary['precision']:.3f}; recall: {summary['recall']:.3f}.
Confidence: {summary['confidence']} (None means classical rules, no learned score).
Median inference-call latency: {summary['median_latency_ms']:.2f} ms.

Intended use: classroom demonstration of a reproducible detection workflow.
Limitations: synthetic domain; small test set; uncalibrated scores; no real-camera validation.
Threshold selection: fixed classroom default; replace with a validation-based record if tuned.
Next experiment: collect independent real scenes and establish a fresh held-out test split.
Artifacts: summary.json, per_image_results.csv, predictions.png, environment.json.
"""
(OUTPUT / "model_report.md").write_text(report, encoding="utf-8")
print(report)
print("Saved to:", OUTPUT)


## Demonstrate your project in ten minutes

1. **Problem (1 min):** define the objects and one success criterion.
2. **Data (2 min):** show labels, negatives and group splits.
3. **Model (2 min):** explain your baseline and training choice.
4. **Evidence (3 min):** run inference, show failures and report measured metrics.
5. **Handoff (2 min):** show exported artifacts and explain remaining limitations.

**Extension challenge:** compare classical, YOLO and Faster R-CNN on the same validation
split; select one using accuracy and measured latency; then run a fresh final test.
Repeat on your own annotated photos with grouped splits. Keep each experiment's seed,
configuration and dataset version. Quantization, transformer detectors, distributed
training and production monitoring are follow-on subjects beyond this course's core.

**Answer/checkpoint:** a stronger model is not automatically a better deployment choice.
Pick using the requirements and measured tradeoffs, and communicate uncertainty from
limited data. Archive a failed example and propose a testable next experiment.

Reference: [Ultralytics validation metrics](https://docs.ultralytics.com/modes/val/).
